# Skin Condition Detector — 22-Class Training (Kaggle)

Train **one model for all 22 conditions** in a single notebook run (Run All → download → done).

## Fastest path (~45–90 min GPU)

1. Open [Skin Disease Dataset](https://www.kaggle.com/datasets/pacificrm/skindiseasedataset)
2. Click **New Notebook** on **that dataset page** (critical — attaches the data)
3. Copy/paste these cells into the new notebook (or upload this file)
4. **Settings** → **Accelerator** → **GPU T4 x2**
5. **Run All** → download `skin_model.pth` from the **Output** tab

> **Got `Dataset not found`?** The dataset is not attached. Use step 2 — don't create a blank notebook first.

Dataset layout is `train/ClassName/*.jpg` and `test/ClassName/*.jpg` — the notebook scans both automatically.

Set `QUICK_MODE = True` in cell 1 for a faster (~25 min) smoke test.

In [ ]:
import os
import re
import copy
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# True = fewer epochs for a quick test (~25 min). False = full training (~60–90 min).
QUICK_MODE = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'QUICK_MODE: {QUICK_MODE}')

CANONICAL_CODES = [
    'acne', 'actinic_keratosis', 'benign_tumors', 'bullous', 'candidiasis',
    'drug_eruption', 'eczema', 'infestations_bites', 'lichen', 'lupus',
    'moles', 'psoriasis', 'rosacea', 'seborrheic_keratoses', 'skin_cancer',
    'sun_damage', 'tinea', 'normal', 'vascular_tumors', 'vasculitis',
    'vitiligo', 'warts',
]

FOLDER_TO_CODE = {
    'acne': 'acne',
    'actinic_keratosis': 'actinic_keratosis',
    'benign_tumors': 'benign_tumors',
    'bullous': 'bullous',
    'candidiasis': 'candidiasis',
    'drug_eruption': 'drug_eruption',
    'eczema': 'eczema',
    'infestations_bites': 'infestations_bites',
    'infestations': 'infestations_bites',
    'lichen': 'lichen',
    'lupus': 'lupus',
    'moles': 'moles',
    'psoriasis': 'psoriasis',
    'rosacea': 'rosacea',
    'seborrheic_keratoses': 'seborrheic_keratoses',
    'seborrh_keratoses': 'seborrheic_keratoses',
    'skin_cancer': 'skin_cancer',
    'sun_sunlight_damage': 'sun_damage',
    'sun_damage': 'sun_damage',
    'tinea': 'tinea',
    'unknown_normal': 'normal',
    'normal': 'normal',
    'vascular_tumors': 'vascular_tumors',
    'vasculitis': 'vasculitis',
    'vitiligo': 'vitiligo',
    'warts': 'warts',
}

INPUT_DIR = Path('/kaggle/input')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLIT_FOLDERS = {'train', 'test', 'val', 'validation', 'training', 'testing'}

if INPUT_DIR.exists():
    print('Mounted inputs:', [p.name for p in INPUT_DIR.iterdir()])
    for mount in INPUT_DIR.iterdir():
        if mount.is_dir():
            children = [c.name for c in mount.iterdir()][:8]
            print(f'  {mount.name}/ -> {children}')
else:
    print('WARNING: /kaggle/input does not exist — are you on Kaggle?')


def normalize_folder(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')


def folder_to_code(name: str) -> str:
    return FOLDER_TO_CODE.get(normalize_folder(name), normalize_folder(name))


def code_from_image_path(image_path: Path) -> str | None:
    """Match class from nearest folder name in the path (e.g. .../train/Acne/img.jpg)."""
    skip = SPLIT_FOLDERS | {'input', 'kaggle', 'working', 'data'}
    for part in reversed(image_path.parts):
        if part.lower() in skip:
            continue
        if part.startswith('skin') and 'disease' in part.lower():
            continue  # skip mount folder like skindiseasedataset
        code = folder_to_code(part)
        if code in CANONICAL_CODES:
            return code
    return None


def scan_pacificrm_images() -> list[dict]:
    """Find all PacificRM images anywhere under /kaggle/input."""
    rows = []
    skipped_ham = 0
    unknown_folders = set()

    if not INPUT_DIR.exists():
        return rows

    for image_path in INPUT_DIR.rglob('*'):
        if not image_path.is_file():
            continue
        if image_path.suffix.lower() not in IMAGE_EXTS:
            continue
        if 'ham10000' in str(image_path).lower():
            skipped_ham += 1
            continue

        code = code_from_image_path(image_path)
        if code:
            rows.append({
                'image_path': str(image_path),
                'code': code,
                'source': 'pacificrm',
            })
        else:
            unknown_folders.add(image_path.parent.name)

    print(f'PacificRM images found: {len(rows)}')
    if rows:
        print('Sample path:', rows[0]['image_path'])
    if unknown_folders and not rows:
        print('Folders with images but no class match:', sorted(unknown_folders)[:10])
    if skipped_ham:
        print(f'(Skipped {skipped_ham} HAM10000 images)')
    return rows

In [ ]:
rows = scan_pacificrm_images()

if not rows:
    hint = """
    No images found. Fix:
    1. Open https://www.kaggle.com/datasets/pacificrm/skindiseasedataset
    2. Click NEW NOTEBOOK on that page (attaches dataset automatically)
       OR in your notebook: right sidebar → Add Input → search "Skin Disease Dataset"
    3. Re-run from the top

    Expected layout (either works):
      .../train/Acne/*.jpg
      .../test/Eczema/*.jpg
      OR .../Acne/*.jpg
    """
    raise FileNotFoundError(hint)

# --- HAM10000 (optional): merges into same 22 classes if you added kmader as input ---
HAM_TO_CANONICAL = {
    'akiec': 'actinic_keratosis',
    'bcc': 'skin_cancer',
    'bkl': 'seborrheic_keratoses',
    'df': 'benign_tumors',
    'mel': 'skin_cancer',
    'nv': 'moles',
    'vasc': 'vascular_tumors',
}

metadata_files = list(INPUT_DIR.rglob('HAM10000_metadata.csv')) if INPUT_DIR.exists() else []
if metadata_files:
    meta_path = metadata_files[0]
    ham_dir = meta_path.parent
    ham_df = pd.read_csv(meta_path)
    image_map = {
        os.path.splitext(os.path.basename(p))[0]: p
        for p in glob(str(ham_dir / '**' / '*.jpg'), recursive=True)
    }
    ham_added = 0
    for _, row in ham_df.iterrows():
        dx = row.get('dx')
        if dx not in HAM_TO_CANONICAL:
            continue
        img_path = image_map.get(row['image_id'])
        if img_path:
            rows.append({
                'image_path': img_path,
                'code': HAM_TO_CANONICAL[dx],
                'source': 'ham10000',
            })
            ham_added += 1
    print(f'HAM10000 images merged: {ham_added}')
else:
    print('HAM10000 not attached (optional)')

df = pd.DataFrame(rows)
code_to_idx = {code: i for i, code in enumerate(CANONICAL_CODES)}
df['label'] = df['code'].map(code_to_idx)
df = df.dropna(subset=['label']).reset_index(drop=True)

print(f'Total images: {len(df)}')
present = sorted(df['code'].unique())
missing = [c for c in CANONICAL_CODES if c not in present]
print(f'Classes found: {len(present)} / {len(CANONICAL_CODES)}')
if missing:
    print('Missing from dataset (no training images):', missing)
print(df['code'].value_counts())

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class SkinDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        return self.transform(image), int(row['label'])

BATCH_SIZE = 32
train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)
train_loader = DataLoader(SkinDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(SkinDataset(val_df, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

In [ ]:
def build_model(num_classes=22):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_classes))
    return model

def train_epochs(model, loader, criterion, optimizer, epochs):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f'Epoch {epoch+1}/{epochs} — loss: {running_loss/len(loader):.4f}')

EPOCHS_STAGE1 = 4 if QUICK_MODE else 8
EPOCHS_STAGE2 = 6 if QUICK_MODE else 15
print(f'Training: stage1={EPOCHS_STAGE1} epochs, stage2={EPOCHS_STAGE2} epochs')

model = build_model(len(CANONICAL_CODES)).to(DEVICE)
criterion = nn.CrossEntropyLoss()

for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
train_epochs(model, train_loader, criterion, optimizer, EPOCHS_STAGE1)

for param in model.parameters():
    param.requires_grad = True
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
train_epochs(model, train_loader, criterion, optimizer, EPOCHS_STAGE2)

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        preds = model(images).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_class_ids = list(range(len(CANONICAL_CODES)))

print(f'Validation accuracy: {(all_preds == all_labels).mean():.2%}')
print(classification_report(
    all_labels,
    all_preds,
    labels=all_class_ids,
    target_names=CANONICAL_CODES,
    zero_division=0,
))

missing_in_val = set(all_class_ids) - set(np.unique(all_labels))
if missing_in_val:
    print('Note: no validation images for:',
          [CANONICAL_CODES[i] for i in sorted(missing_in_val)])

In [ ]:
OUTPUT_PATH = '/kaggle/working/skin_model.pth'
checkpoint = {
    'model_state_dict': model.state_dict(),
    'class_codes': CANONICAL_CODES,
    'dataset': 'pacificrm/skindiseasedataset',
    'num_classes': len(CANONICAL_CODES),
}
torch.save(checkpoint, OUTPUT_PATH)
print(f'Saved to {OUTPUT_PATH} — download from Output tab')